# Tutorial: Fitting data with uncertainties

Data fitting becomes a lot more meaningful when you account for the fact that real-world measurements aren't perfect. In session 7 we performed a simple "best-fit line" through data points using least the least square method, which minimises the sum of the squared differences (residuals) between the actual observed values ($y_i$) and the values predicted by the fit ($\hat{y_i}$)

$$
\sum_{i=1}^{n} (y_i - \hat{y_i})^2.
$$

## Task 1

In `diffusion.csv` you have a set of data from a diffusion study.

If you look at this data you will see the diffusion values are recorded in $\mathrm{cm^2s^{−1}}$ and the temperature is in Fahrenheit. We are going to need to convert these into SI units i.e. $\mathrm{m^2s^{−1}}$ and $\mathrm{K}$. Hopefully you have ready
made examples that you adapt for this from last week.

The movement of atoms in a solid is given by the Arrhenius equation:

$$ 
D = D_0 \exp\left(-\frac{E_A}{k_bT}\right),
$$

where $D_0$ is material dependent constant, $E_A$ is activation energy, $k_B$ is the Boltzmann constant, and $T$ is the temperature in Kelvin and returns the diffusion coefficient $D$.

By taking the natural log of this equation we obtain:

$$
\ln D = \ln D_0 -  \left(\frac{E_A}{k_B T}\right).
$$

1. Take the natural log of the $D$ values and find $1/T$ values.

2. Plot a graph of $\ln D$ (y-axis) and $1/T$ (x-axis). Remember to label your axis. 

3. Perform a linear fit (this is a 1st order polynomial, session 7) and extract a value for $E_A$ (the gradient) and $\ln D_0$ (the constant).

4. Calculate the errors on the $\ln D$ and $1/T$ values following your propagation of error rules. You can use your functions from session 10.

## Weighted Least Squares ($\chi^2$)

As seen in Task 1 we did not consider the uncertainties when fitting the data. When you have uncertainties (errors) for your $y$-data, you want the fit to prioritise points with small errors and care less about points with large errors.

Instead of minimising the sum of squared residuals, we minimise the Chi-squared ($\chi^2$):

$$
\sum_{i=1}^{n} \left(\frac{y_i - \hat{y_i}}{\sigma_i}\right)^2,
$$

where $\sigma_i$​ represents the uncertainty of each data point. Therefore, data points with large uncertainties are weighted less (weighted least square).

## Task 2

In Task 1 you have hopefully used  <a href="https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html" target="_blank">`np.polyfit()`</a>. If you follow the link or use the help, we will see there is an option to provide Weights as an argument to the polyfit function (`w_i = 1/sigma_i`, see also $\chi^2$ equation). 

**Note:** you can only provide weights for your $y$-data (dependent variable). Next week we will also consider uncertainties on the $x$-data (independent variable).

1. Use `np.polyfit()` with weights on $\ln D$.

2. Compare the value you have extracted for $E_A$ and $\ln D_0$ to the values you found in Task 1.

## Task 3

Now we have look at the uncertainties on our fitting parameters. To do so we need to access the covariance matrix:

1. `cov=True`: the covariance is scaled. The weights are presumed to be unreliable except in a relative sense and everything is scaled such that the reduced $\chi^2$ is unity.

2. `cov='unscaled'`: if $\sigma$ is a representative estimate of the uncertainty.

If you trust that your errors ($\sigma$) are a true representation of the uncertainties, you set `p, pcov = np.polyfit(x_data, y_data, w=1/sigma_y, cov='unscaled')`.

Above, we stored the covariance matrix in the variable `pcov`. The variance of the parameters are on the diagonal of the covariance matrix => `perr = np.sqrt(np.diag(pcov))`. The covariance matrix is a symmetric matrix ($M=M^{\mathrm{T}}$). Non zero values which are not located on the diagonal mean that parameters are not independent and correlate.

Extract the uncertainty for $E_A$ and $\ln D_0$.

## Least Square and Weighted Least Square with user defined functions

So far we have used polynomials for curve fitting, but we can also define our own fitting function (model). The first argument or first function parameter must be the independent variable. The following entries are the fitting parameters. For example:

In [ ]:
def model_function(x, m, c):
    """
    x: independent variable
    m, c: parameters with gradient m and constant c 
    """
    y = m*x + c
    return y

To fit the function to the data the Scipy function <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html" target="_blank">`scipy.optimize.curve_fit()`</a> is used.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

# Synthetic data with noise and varying uncertainties
x_data = np.linspace(0, 10, 20)
y_true = model_function(x_data, 2.5, 1.3)
y_err = 0.5 + 0.5 * np.random.rand(20) # Random uncertainties
y_data = y_true + np.random.normal(0, y_err) # y values vary within in a normal distribution with standard deviation equal to y_err

# Fit the data
# sigma=y_err
popt, pcov = curve_fit(model_function, x_data, y_data, sigma=y_err, absolute_sigma=True)

# Extract results
print("Covariance matrix", pcov)
perr = np.sqrt(np.diag(pcov)) # Standard deviations of the parameters

print(f"Slope: {popt[0]:.2f} ± {perr[0]:.2f}")
print("Constant: {:.2f} ± {:.2f}".format(popt[1], perr[1]))

**Note:** 
1. Unlike for `np.polyfit()` the standard deviation is given (`sigma=y_err`) instead of the weight.
2. To obtain the `unscaled` covariance matrix you need to set `absolute_sigma=True`.

Always look up the function documentation before using a function.

## Task 4

Using `curve_fit()` and the exponential form of the Arrhenius equation find $D_0$ and $E_A$ with uncertainties,

$$ 
D = D_0 \exp\left(-\frac{E_A}{k_bT}\right).
$$

Values for $D_0$ and $E_A$ are small (Task 2 and 3). To help `curve_fit()` finding a solution, we can provide <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html" target="_blank">initial guesses</a> for $D_0$ and $E_A$, e.g. `p0=[1e-8, 1e-19]`:

```python
popt, pcov = curve_fit(arrhenius_equation, T, D, sigma=D_err, 
        p0=[1e-8, 1e-19], absolute_sigma=True)
```
Compare the values you have extracted for $D_0$ and $E_A$ including uncertainties to the values you found in Task 3.

In [ ]:
from scipy.optimize import curve_fit

# define arrhenius equation

# least square fit with curve_fit() with  initial guesses

# Extract parameters with uncertainties and compare.

Note: Using the exponential form of the Arrhenius equation we do not need to perform error propagations when calculating $D_0$ and $E_A$. However, we need to provide initial guesses of the parameters, as it is numerically less robust to fit than the logarithmic version of the Arrhenius equation.

## Task 5

A data set is listed in `task5.csv`. The data needs to be fitted with the following relationship:

$$
y = ax^3 + bx.
$$

Perform the fit and determine the values of $a$ and $b$ including uncertainties. 

Plot the $x$ and $y$ data with $y$ error bars using the <a href="https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.errorbar.html" target="_blank">`plt.errorbar()`</a> function. Follow the link and using the documentation to do this.

Use the values of $a$ and $b$ to plot a graph of your fit (i.e. values of $y$ calculated from the equation at those $x$ values using your $a$ and $b$ constants).

In [ ]:
import matplotlib.pyplot as plt

## Task 6

`task6.csv` contains a set of data for a loading experiment to record the true stress and strain of a material.

The data collected has the force applied, the area of the sample and the length recorded. The original length of the sample was $1.002~\mathrm{m}$.

Calculate the true stress and true strain data using the values in `task6.csv`. 

Note: You should be able to find out how to do this from your CMB120 notes. Plot the data set.

Unfortunately, there was a problem with the data collection and a surge in the data caused some problems with its collection. This has led to a number of the data points being recorded incorrectly.

Fortunately, you have realised the problem occurred with every 6th data point collected so you simply need to remove the 6th, 12th, 18th etc. data points. 

Perform this operation with Python. Then plot the data with the problem points removed.